# Intro to Python for Data Analysis

Welcome! This notebook walks you through the basics of using Python to analyze experimental data. By the end you'll be able to:

1. Set up Python and a coding environment (IDE)
2. Import the core data-science packages: **NumPy**, **pandas**, and **Matplotlib**
3. Load a CSV file of experimental data into a pandas **DataFrame**
4. Plot an animal's trajectory and compute where it spent its time
5. Repeat the analysis for many runs with a **loop** and collect the results in a new table
6. Compare runs, and save your results and figures

**The example dataset.** The `data/` folder next to this notebook holds tracking data from three (simulated) behavioral runs. In each 5-minute run, a mouse explored a 60 × 40 cm arena while a camera recorded its position 10 times per second.

| File | What's in it |
|---|---|
| `data/run1.csv`, `data/run2.csv`, `data/run3.csv` | one row per video frame: `time_s`, `x_cm`, `y_cm` |
| `data/metadata.csv` | one row per run: animal, date, frame rate, arena size, and the **x-coordinate of the arena's midpoint** (it shifts slightly between runs depending on where the camera sat) |

**Our question:** *What percent of the time did each animal spend on the left vs. right side of the arena?*

> **How to use this notebook:** Read each text cell, then run the code cell below it. Try changing things and re-running — you can't break anything. Cells marked **✏️ Try it** are short exercises.

---
## 1. Setting up Python and an IDE

An **IDE** (Integrated Development Environment) is the program you write code in. We recommend **VS Code**, a free editor that runs Jupyter notebooks like this one. You only need to do this setup once.

### Step 1 — Install Python (via Miniforge)

We'll install Python with **Miniforge**, which gives you Python plus the `conda` tool for managing packages and environments.

1. Go to <https://github.com/conda-forge/miniforge> and download the installer for your operating system.
2. Run the installer and accept the defaults.
   - **Windows:** afterwards, open the *Miniforge Prompt* from the Start menu.
   - **Mac/Linux:** open the *Terminal* app.
3. Check it worked by typing:
   ```bash
   python --version
   ```
   You should see something like `Python 3.12.x`.

### Step 2 — Create an environment and install packages

An **environment** is an isolated box of Python packages for a project, so different projects don't interfere with each other. In your terminal, run:

```bash
conda create -n project_env python=3.12 numpy pandas matplotlib scipy ipykernel
conda activate project_env
```

- `conda create -n project_env ...` makes an environment named `project_env` with the packages we need.
- `conda activate project_env` switches into it. You'll do this each time you open a new terminal.

> Already have Python some other way? You can instead run `pip install numpy pandas matplotlib scipy ipykernel`.

### Step 3 — Install VS Code

1. Download and install VS Code from <https://code.visualstudio.com>.
2. Open VS Code, click the **Extensions** icon in the left sidebar (four little squares), and install the **Python** and **Jupyter** extensions (both by Microsoft).

### Step 4 — Open this notebook

1. In VS Code, choose **File → Open Folder…** and open the folder containing this notebook and the `data` folder.
2. Click `tutorial.ipynb` to open it.
3. Click **Select Kernel** in the top-right corner and choose the `project_env` environment.

### Notebook basics

- A notebook is made of **cells**. Text cells (like this one) explain things; **code cells** contain Python.
- Run a code cell by clicking it and pressing **Shift + Enter**.
- Output appears directly below the cell.
- Cells share memory: a variable created in one cell can be used in later cells. **Run cells in order, top to bottom.** If things get confusing, use **Restart** (top toolbar) and run everything again.

Run the cell below. If it prints a version number with no errors, you're set up!

In [ ]:
import sys
print("Python version:", sys.version)

### A 5-minute tour of Python

Before working with data, here are the handful of Python ideas you'll see throughout this notebook.

**Variables** store values. Text after a `#` is a *comment* - Python ignores it.

In [ ]:
n_runs = 3               # an integer (whole number)
midpoint = 30.4          # a float (decimal number)
animal = "mouse_01"      # a string (text) goes in quotes

print(n_runs, midpoint, animal)

**Lists** hold several values in order. Python starts counting at **0**, so the first item is `[0]`.

In [ ]:
x_positions = [12.5, 18.0, 33.2, 41.7]

print(x_positions[0])      # first item
print(len(x_positions))    # how many items
print(sum(x_positions) / len(x_positions))  # average

**Dictionaries** store values under names (*keys*), which is handy for grouping related results together.

In [ ]:
result = {"run_id": "run1", "pct_left": 74.3}
print(result["pct_left"])

**`for` loops** repeat a block of code once for each item in a list. The indented lines are the "body" of the loop. Indentation matters in Python!

In [ ]:
for run in ["run1", "run2", "run3"]:
    print("Now analyzing", run)
print("Done!")   # not indented, so this runs once, after the loop

**`if` statements** run code only when a condition is `True`. Note the **double** equals `==` for "is equal to" (a single `=` assigns a variable).

In [ ]:
for x in x_positions:
    if x < midpoint:
        print(x, "is on the left")
    else:
        print(x, "is on the right")

**Functions** take inputs (in parentheses) and return outputs. `print`, `len`, and `sum` are built-in functions. You can write your own with `def` — a function is a reusable recipe, which we'll lean on heavily in Section 5.

In [ ]:
def percent(part, whole):
    # Return part as a percentage of whole.
    return 100 * part / whole

percent(3, 4)

> Notice the last cell had no `print` — a notebook automatically displays the value of the **last line** in a cell.

**✏️ Try it:** Write a loop that goes through `x_positions` and counts how many are on the left side. *(Hint: make a variable `count = 0` before the loop, and do `count = count + 1` inside the `if`.)*

In [ ]:
# your code here

---
## 2. Importing packages

Python on its own is fairly basic. **Packages** (also called *libraries*) add extra tools written by other people. The core data-science packages are:

| Package | What it's for | Usual nickname |
|---|---|---|
| **NumPy** | fast math on arrays of numbers | `np` |
| **pandas** | tables of data (like a spreadsheet) | `pd` |
| **Matplotlib** | making plots | `plt` |
| **SciPy** | statistics and scientific tools | `stats` (for `scipy.stats`) |

We bring a package into our notebook with `import`. The `as` part gives it a short nickname so we don't have to type the full name every time. These nicknames are a universal convention — you'll see them in almost every example online.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

There are a few styles of import:

- `import numpy as np` — import the whole package; use it as `np.something`.
- `from scipy import stats` — import just one piece (a *submodule*) of a package.
- `from math import sqrt` — import one specific function, which you can then call directly.

Once imported, you use a package's functions with a dot: `package.function()`.

In [ ]:
from math import sqrt

values = np.array([2.1, 3.4, 2.8, 4.0])   # a NumPy array

print("mean:", np.mean(values))
print("standard deviation:", np.std(values, ddof=1))  # ddof=1 gives the *sample* SD
print("square root of 16:", sqrt(16))

> **Getting help:** put a `?` after any function name and run the cell (e.g. `np.mean?`) to see its documentation. Searching the web for *"pandas how to …"* is also completely normal — professionals do it constantly.

> **`ModuleNotFoundError`?** That means the package isn't installed in the environment your notebook is using. Check that you selected the `labpy` kernel (Section 1, Step 4).

---
## 3. Loading data

A **CSV** (comma-separated values) file is a plain-text table: each line is a row, and commas separate the columns. You can open any of the files in `data/` in Excel or a text editor to see what they look like.

`pd.read_csv()` reads a CSV into a **DataFrame** — pandas' version of a spreadsheet table, with named columns and numbered rows.

### The metadata

Let's start with the metadata, since it tells us about every run.

In [ ]:
meta = pd.read_csv("data/metadata.csv")
meta

> **`FileNotFoundError`?** Python looks for files relative to the notebook's folder. `"data/metadata.csv"` means "the file `metadata.csv` inside the `data` folder next to this notebook." Check that the folder is there, or give the full path (e.g. `"/Users/me/lab/data/metadata.csv"`).

**Selecting data.** Grab one column with square brackets and its name. A single column is called a **Series**.

In [ ]:
meta["midpoint_x_cm"]

**Filtering rows.** Put a True/False condition in the brackets to keep only the rows where it's `True`. Then `.iloc[0]` picks out the first (here, the only) matching row.

In [ ]:
run1_info = meta[meta["run_id"] == "run1"].iloc[0]
run1_info

In [ ]:
midpoint = run1_info["midpoint_x_cm"]
frame_rate = run1_info["frame_rate_hz"]
print("Run 1 midpoint:", midpoint, "cm   frame rate:", frame_rate, "Hz")

### One run's trajectory

Now load the trajectory for run 1. The file name comes from the metadata, so we don't have to type it by hand.

In [ ]:
traj = pd.read_csv("data/" + run1_info["filename"])
traj.head()   # first 5 rows (try traj.head(10) or traj.tail())

In [ ]:
print("shape (rows, columns):", traj.shape)
print("column names:", list(traj.columns))

3,000 rows = 300 seconds × 10 frames per second. ✔️

### Summarizing the data

`.describe()` gives summary statistics for every numeric column — a quick sanity check. For example, all x values should fall between 0 and 60 cm (the arena width).

In [ ]:
traj.describe()

### Checking for missing values

Tracking software sometimes loses the animal for a few frames (e.g. when it is hidden by a shadow). Those frames show up as empty cells in the CSV, which pandas reads as `NaN` ("not a number"). `.isna().sum()` counts them per column. Run 1 is clean, but we'll meet a run with gaps later.

In [ ]:
traj.isna().sum()

**✏️ Try it:** Make a DataFrame containing only the frames from the first 10 seconds of run 1. *(Hint: filter on the `time_s` column.)*

In [ ]:
# your code here

---
## 4. Analyzing a single run

### 4a. Plotting the trajectory

Matplotlib is the standard Python plotting library. The usual pattern is:

```python
fig, ax = plt.subplots()   # make a figure (fig) with one set of axes (ax)
ax.<plot type>(...)        # draw on the axes
ax.set_xlabel(...)         # label things
plt.show()                 # display it
```

**Always label your axes, with units!** Here we draw the animal's path as a line, and mark the midpoint with a dashed vertical line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))   # figsize is (width, height) in inches

ax.plot(traj["x_cm"], traj["y_cm"], linewidth=0.7)
ax.axvline(midpoint, color="red", linestyle="--", label="midpoint")

ax.set_xlim(0, 60)
ax.set_ylim(0, 40)
ax.set_aspect("equal")            # so 1 cm in x looks the same as 1 cm in y
ax.set_xlabel("x position (cm)")
ax.set_ylabel("y position (cm)")
ax.set_title("Run 1 trajectory")
ax.legend()
plt.show()

It's also useful to see position **over time**. Plotting x against time shows when the animal crossed sides:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(traj["time_s"], traj["x_cm"], linewidth=0.7)
ax.axhline(midpoint, color="red", linestyle="--")
ax.set_xlabel("Time (s)")
ax.set_ylabel("x position (cm)")
ax.set_title("Run 1: x position over time (left side is below the red line)")
plt.show()

An **occupancy heatmap** shows where the animal spent the most time. `hist2d` splits the arena into a grid of bins and counts the frames in each:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
counts, xedges, yedges, image = ax.hist2d(traj["x_cm"], traj["y_cm"], bins=[30, 20],
                                          range=[[0, 60], [0, 40]], cmap="viridis")
ax.axvline(midpoint, color="white", linestyle="--")
ax.set_aspect("equal")
ax.set_xlabel("x position (cm)")
ax.set_ylabel("y position (cm)")
ax.set_title("Run 1 occupancy")
fig.colorbar(image, ax=ax, label="Number of frames")
plt.show()

### 4b. Which side is the animal on? Creating new columns

To make a new column, assign to a new column name. Comparisons and math on columns happen **row by row** automatically — no loop needed. Here we make a True/False column that says whether the animal was on the left in each frame:

In [ ]:
traj["is_left"] = traj["x_cm"] < midpoint
traj.head()

It's often nicer to have a readable label. `np.where(condition, value_if_true, value_if_false)` picks between two values for every row:

In [ ]:
traj["side"] = np.where(traj["is_left"], "left", "right")
traj.head()

### 4c. Percent of time on each side

`.value_counts()` counts how many frames have each label. Adding `normalize=True` gives fractions instead of counts.

In [ ]:
traj["side"].value_counts()

In [ ]:
traj["side"].value_counts(normalize=True) * 100

Because the camera records at a steady rate, **percent of frames = percent of time**. To get seconds instead, divide the number of frames by the frame rate.

There's also a handy shortcut: Python treats `True` as 1 and `False` as 0, so the **mean** of a True/False column is the fraction of `True` values.

In [ ]:
pct_left = traj["is_left"].mean() * 100
time_left_s = traj["is_left"].sum() / frame_rate

print(f"Run 1: {pct_left:.1f}% of time on the left ({time_left_s:.0f} s of {len(traj) / frame_rate:.0f} s)")

> The `f"..."` is an **f-string**: anything inside `{ }` gets filled in, and `:.1f` means "round to 1 decimal place."

### 4d. One more measure: distance traveled

Animals that spend lots of time on one side might also just be moving less. Let's compute how far the animal traveled — another example of building new columns from existing ones.

`.diff()` subtracts each row from the row before it, giving the change in position between frames. Then the Pythagorean theorem gives the distance moved in each frame:

In [ ]:
traj["step_cm"] = np.sqrt(traj["x_cm"].diff()**2 + traj["y_cm"].diff()**2)
traj.head()

The first row is `NaN` because there's no previous frame to compare with. `.sum()` skips `NaN` values automatically.

In [ ]:
total_distance = traj["step_cm"].sum()
mean_speed = total_distance / (len(traj) / frame_rate)
print(f"Total distance: {total_distance:.0f} cm   Mean speed: {mean_speed:.1f} cm/s")

**✏️ Try it:** Compute the percent of time run 1's animal spent in the **top half** of the arena (y greater than 20 cm).

In [ ]:
# your code here

---
## 5. Analyzing every run with a loop

We have 3 runs, but a real experiment might have 30 or 300. Copying and pasting the code above for each run would be slow and error-prone (it's easy to forget to change the midpoint!). Instead:

1. Wrap the analysis in a **function** so it's written only once.
2. **Loop** over the rows of the metadata table, calling the function on each run.
3. Collect each run's results and turn them into a new DataFrame.

### 5a. Wrap the analysis in a function

This function takes a trajectory DataFrame plus the run's midpoint and frame rate, and returns a dictionary of results. It contains the same steps as Section 4, plus one new one: **dropping frames with missing positions**.

Why does that matter? A comparison with `NaN` is always `False`, so `NaN < midpoint` would silently count every lost frame as "right side." `.dropna()` removes rows with missing values so they don't bias the result.

In [ ]:
def analyze_run(traj, midpoint, frame_rate):
    # Compute side preference and movement measures for one run.
    n_missing = traj["x_cm"].isna().sum()
    traj = traj.dropna(subset=["x_cm", "y_cm"])    # remove frames where tracking was lost

    is_left = traj["x_cm"] < midpoint
    steps = np.sqrt(traj["x_cm"].diff()**2 + traj["y_cm"].diff()**2)
    duration_s = len(traj) / frame_rate

    return {
        "n_frames": len(traj),
        "n_missing_frames": int(n_missing),
        "pct_left": float(is_left.mean() * 100),
        "pct_right": float((~is_left).mean() * 100),   # ~ flips True <-> False
        "time_left_s": float(is_left.sum() / frame_rate),
        "distance_cm": float(steps.sum()),
        "mean_speed_cm_s": float(steps.sum() / duration_s),
    }

> `int()` and `float()` convert pandas/NumPy numbers into plain Python numbers, which just makes the printed dictionary easier to read.

First, check that it gives the same answer as before for run 1:

In [ ]:
analyze_run(traj, midpoint, frame_rate)

### 5b. Loop over the runs

`meta.iterrows()` steps through the metadata one row at a time. Each time through the loop, `row` holds one run's info, so we can look up its filename, midpoint, and frame rate.

We also save each trajectory in a dictionary called `trajectories` so we can plot them all later.

In [ ]:
results = []        # an empty list we'll add to
trajectories = {}   # an empty dictionary we'll fill: run_id -> DataFrame

for index, row in meta.iterrows():
    run_traj = pd.read_csv("data/" + row["filename"])
    run_result = analyze_run(run_traj, row["midpoint_x_cm"], row["frame_rate_hz"])

    run_result["run_id"] = row["run_id"]   # label the result so we know which run it came from
    results.append(run_result)              # add it to the list
    trajectories[row["run_id"]] = run_traj

    print(f"{row['run_id']}: {run_result['pct_left']:.1f}% left, "
          f"{run_result['n_missing_frames']} missing frames")

Notice that run 2 had 12 frames where tracking was lost — good thing our function handles that.

### 5c. Build a summary DataFrame

A list of dictionaries converts straight into a DataFrame: each dictionary becomes a row and each key becomes a column.

In [ ]:
summary = pd.DataFrame(results)
summary

It would be nice to have the metadata (animal, date, …) in the same table. `.merge()` joins two DataFrames by matching values in a shared column — like a VLOOKUP in Excel.

In [ ]:
summary = meta.merge(summary, on="run_id")
summary

That's a lot of columns. Select just the ones you care about, and round for readability:

In [ ]:
summary[["run_id", "animal_id", "pct_left", "pct_right", "distance_cm", "mean_speed_cm_s"]].round(1)

**✏️ Try it:** Add a `"pct_top"` entry (percent of time with y > 20 cm) to the `analyze_run` function, then re-run the function cell and the loop. Does your new column show up in `summary`?

In [ ]:
# your code here (or edit the function above)

---
## 6. Comparing runs

### 6a. Side preference across runs

A bar chart of percent time on the left, with a dashed line at 50% (no preference):

In [ ]:
fig, ax = plt.subplots()
ax.bar(summary["run_id"], summary["pct_left"], color="tab:blue")
ax.axhline(50, color="gray", linestyle="--", label="no preference (50%)")
ax.set_ylim(0, 100)
ax.set_xlabel("Run")
ax.set_ylabel("Time on left side (%)")
ax.set_title("Side preference by run")
ax.legend()
plt.show()

A common way to summarize side preference in a single number is a **preference index**, which runs from −1 (always right) through 0 (no preference) to +1 (always left):

$$\text{PI} = \frac{\%\text{left} - \%\text{right}}{100}$$

Since it's just math on existing columns, it's one line:

In [ ]:
summary["preference_index"] = (summary["pct_left"] - summary["pct_right"]) / 100
summary[["run_id", "pct_left", "preference_index"]].round(2)

`.sort_values()` ranks the runs, and `.describe()` summarizes a column across all runs:

In [ ]:
summary.sort_values("pct_left", ascending=False)[["run_id", "animal_id", "pct_left"]]

In [ ]:
summary["pct_left"].describe()

### 6b. All trajectories side by side

`plt.subplots(1, 3)` makes a row of 3 plots. `axes` is then a list of 3 axes, and we loop over the runs and the axes together with `zip`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (index, row) in zip(axes, meta.iterrows()):
    t = trajectories[row["run_id"]]
    ax.plot(t["x_cm"], t["y_cm"], linewidth=0.5)
    ax.axvline(row["midpoint_x_cm"], color="red", linestyle="--")
    pct = summary.loc[summary["run_id"] == row["run_id"], "pct_left"].iloc[0]
    ax.set_title(f"{row['run_id']} ({row['animal_id']}): {pct:.0f}% left")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 40)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cm)")

axes[0].set_ylabel("y (cm)")
plt.tight_layout()   # stop the subplots from overlapping
plt.show()

### 6c. Does preference change over the session?

So far we've summarized each run with one number. But maybe animals explore evenly at first and develop a preference later. To check, we'll compute percent-left in each minute of each run.

First, stack all three trajectories into one long DataFrame with `pd.concat`, adding a `run_id` column so we can tell them apart. This "long" format — one row per observation, with a column saying which group it belongs to — is the most flexible way to store data in pandas.

In [ ]:
all_runs = []
for index, row in meta.iterrows():
    t = trajectories[row["run_id"]].dropna().copy()   # .copy() avoids changing the stored original
    t["run_id"] = row["run_id"]
    t["is_left"] = t["x_cm"] < row["midpoint_x_cm"]
    t["minute"] = (t["time_s"] // 60).astype(int) + 1  # // is whole-number division: 0-59 s -> minute 1
    all_runs.append(t)

long_df = pd.concat(all_runs, ignore_index=True)
long_df

Now `.groupby()` does the heavy lifting: split the rows by run **and** minute, then take the mean of `is_left` in each group. Read it as *"group by run and minute, take the is_left column, and compute its mean."*

`.unstack()` then pivots the result into a table with one row per minute and one column per run.

In [ ]:
by_minute = long_df.groupby(["run_id", "minute"])["is_left"].mean() * 100
by_minute = by_minute.unstack("run_id")
by_minute.round(1)

In [ ]:
fig, ax = plt.subplots()
for run_id in by_minute.columns:
    ax.plot(by_minute.index, by_minute[run_id], marker="o", label=run_id)

ax.axhline(50, color="gray", linestyle="--")
ax.set_ylim(0, 100)
ax.set_xticks(by_minute.index)
ax.set_xlabel("Minute of session")
ax.set_ylabel("Time on left side (%)")
ax.set_title("Side preference over time")
ax.legend()
plt.show()

### 6d. A note on statistics

Is the group's preference for the left *significantly* different from 50%? With several animals, a **one-sample t-test** answers this: it asks whether the mean of a set of values differs from a fixed number. The **p-value** is the probability of seeing a difference at least this large *if there were really no preference*; a common cutoff is p < 0.05.

In [ ]:
result = stats.ttest_1samp(summary["pct_left"], popmean=50)
print(f"mean = {summary['pct_left'].mean():.1f}%, t = {result.statistic:.2f}, p = {result.pvalue:.3f}")

Two important caveats:

- **The unit of analysis is the animal, not the frame.** It's tempting to treat all 3,000 frames as independent data points, but consecutive frames are nearly identical, so that would wildly overstate your confidence. Compute one number per animal (as we did), then do statistics on those numbers.
- **n = 3 is far too few** to conclude much — here, one animal prefers the left, one the right, and one neither, so of course there's no consistent group effect. The code is the same with 30 animals; just add rows to `metadata.csv`. To compare two groups of animals (e.g. control vs. treated), you'd add a `condition` column to the metadata and use `stats.ttest_ind` on the two groups' `pct_left` values.

We can also check whether side preference is related to how much the animals moved, using a correlation (*r* runs from −1 through 0 to +1). Again, with three points this is just a demonstration:

In [ ]:
r, p = stats.pearsonr(summary["preference_index"].abs(), summary["mean_speed_cm_s"])
print(f"r = {r:.2f}, p = {p:.2f}")

**✏️ Try it:** Make a scatter plot of `mean_speed_cm_s` (x-axis) against `pct_left` (y-axis), with one point per run. Bonus: label each point with its run ID using `ax.text(x, y, label)` inside a loop.

In [ ]:
# your code here

---
## 7. Saving data and plots

It's good practice to keep your **raw data untouched** and save results to a separate folder. Let's make an `outputs` folder next to this notebook (`exist_ok=True` means "don't complain if it already exists").

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

### Saving DataFrames

`.to_csv()` writes a DataFrame to a CSV file. `index=False` stops pandas from adding an extra column of row numbers.

In [ ]:
summary.to_csv("outputs/run_summary.csv", index=False)
by_minute.to_csv("outputs/pct_left_by_minute.csv")   # keep the index here: it holds the minute numbers
print("Saved!")

> Want an Excel file instead? `summary.to_excel("outputs/run_summary.xlsx", index=False)` works too (it needs the `openpyxl` package installed).

### Saving figures

Call `fig.savefig()` **before** `plt.show()`. The file type is set by the extension:

- `.png` — good for slides and quick sharing (`dpi` sets resolution; 300 is publication quality)
- `.pdf` or `.svg` — *vector* formats that stay sharp at any size; best for papers and posters, and editable in Illustrator/Inkscape

`bbox_inches="tight"` trims extra white space around the figure.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(summary["run_id"], summary["pct_left"], color="tab:blue")
ax.axhline(50, color="gray", linestyle="--")
ax.set_ylim(0, 100)
ax.set_xlabel("Run")
ax.set_ylabel("Time on left side (%)")

fig.savefig("outputs/pct_left_by_run.png", dpi=300, bbox_inches="tight")
fig.savefig("outputs/pct_left_by_run.pdf", bbox_inches="tight")
plt.show()

Loops work for saving too. This saves one trajectory plot per run, with the run ID in the filename:

In [ ]:
for index, row in meta.iterrows():
    t = trajectories[row["run_id"]]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(t["x_cm"], t["y_cm"], linewidth=0.5)
    ax.axvline(row["midpoint_x_cm"], color="red", linestyle="--")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 40)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    ax.set_title(row["run_id"])
    fig.savefig(f"outputs/trajectory_{row['run_id']}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)   # close instead of show, so the notebook isn't flooded with plots

os.listdir("outputs")

---
## Wrap-up

You've gone through a complete analysis workflow:

| Step | Key code |
|---|---|
| Import packages | `import pandas as pd` |
| Load data | `df = pd.read_csv("data/file.csv")` |
| Look at it | `df.head()`, `df.describe()`, `df.isna().sum()` |
| Select / filter | `df["col"]`, `df[df["col"] > 5]` |
| New column | `df["is_left"] = df["x_cm"] < midpoint` |
| Percent of time | `df["is_left"].mean() * 100` |
| Handle missing data | `df.dropna()` |
| Reusable analysis | `def analyze_run(...): ... return {...}` |
| Loop over runs | `for index, row in meta.iterrows(): ...` |
| Collect results | `pd.DataFrame(list_of_dicts)`, `meta.merge(summary, on="run_id")` |
| Group summaries | `df.groupby(["run_id", "minute"])["col"].mean()` |
| Statistics | `stats.ttest_1samp(values, 50)`, `stats.ttest_ind(a, b)`, `stats.pearsonr(x, y)` |
| Plot | `fig, ax = plt.subplots()`, `ax.plot/bar/hist2d(...)` |
| Save | `df.to_csv(...)`, `fig.savefig(...)` |

### Next steps

- **Try it on your own data:** export tracking data as CSVs, make a `metadata.csv` describing each run, and point the loop at it.
- **Seaborn** (`import seaborn as sns`) is built on Matplotlib and makes grouped statistical plots in one line, e.g. `sns.barplot(data=summary, x="run_id", y="pct_left")`.
- Helpful references:
  - pandas "10 minutes to pandas": <https://pandas.pydata.org/docs/user_guide/10min.html>
  - Matplotlib gallery (find a plot you like, copy its code): <https://matplotlib.org/stable/gallery/>
  - SciPy statistics reference: <https://docs.scipy.org/doc/scipy/reference/stats.html>